In [100]:
import pandas as pd
import yfinance as yf

def calculate_annual_revenue_growth(tickers):
    results = []
    
    for ticker in tickers:
        try:
            # Fetch the stock data
            stock = yf.Ticker(ticker)
            
            # Get the annual financial data
            financials = stock.financials
            
            if financials.empty or 'Total Revenue' not in financials.index:
                results.append((ticker, None))
                continue
            
            # Extract revenue data for the last two years
            revenue = financials.loc['Total Revenue']
            
            if len(revenue) < 2:
                results.append((ticker, None))
                continue
            
            # Calculate revenue growth
            last_year_revenue = revenue.iloc[0]
            previous_year_revenue = revenue.iloc[1]
            
            if previous_year_revenue != 0:
                growth = (last_year_revenue - previous_year_revenue) / previous_year_revenue * 100
            else:
                growth = float('inf')  # Handle division by zero
            
            results.append((ticker, growth))
        except Exception as e:
            results.append((ticker, None))
            print(f"Error processing {ticker}: {str(e)}")
    
    return pd.DataFrame(results, columns=['Ticker', 'Annual Revenue Growth (%)'])

def convert_ticker_format(ticker):
    """
    Convert ticker from 'EXCHANGE : TICKER' format to Yahoo Finance format.
    
    Args:
    ticker (str): Ticker in 'EXCHANGE : TICKER' format
    
    Returns:
    str: Ticker in Yahoo Finance format
    """
    if ':' not in ticker:
        return ticker  # Already in correct format

    exchange, symbol = ticker.split(':')
    exchange = exchange.strip().upper()
    symbol = symbol.strip().upper()

    # Dictionary of exchange codes and their Yahoo Finance suffixes
    exchange_suffixes = {
        'NYSE': '',  # No suffix for NYSE
        'NASDAQ': '',  # No suffix for NASDAQ
        'BIT': '.MI',  # Milan
        'TYO': '.T',   # Tokyo
        'LSE': '.L',   # London
        'FRA': '.F',   # Frankfurt
        'PAR': '.PA',  # Paris
        'AMS': '.AS',  # Amsterdam
        'TSE': '.TO',  # Toronto
        'SWX': '.SW',  # Switzerland
        'HKG': '.HK',  # Hong Kong
    }

    # Default to no suffix if exchange not found
    suffix = exchange_suffixes.get(exchange, '')
    
    return f"{symbol}{suffix}"

def convert_ticker_format(ticker):
    if ':' not in ticker:
        return ticker

    exchange, symbol = ticker.split(':')
    exchange = exchange.strip().upper()
    symbol = symbol.strip().upper()

    exchange_suffixes = {
        'NYSE': '', 'NASDAQ': '', 'BIT': '.MI', 'TYO': '.T', 'LSE': '.L',
        'FRA': '.F', 'PAR': '.PA', 'AMS': '.AS', 'TSE': '.TO', 'SWX': '.SW', 'HKG': '.HK',
    }

    suffix = exchange_suffixes.get(exchange, '')
    return f"{symbol}{suffix}"

def analyze_stock_growth(df):
    """
    Analyze stock growth based on annual revenue for a DataFrame of tickers.
    
    Args:
    df (pandas.DataFrame): DataFrame with a 'Ticker' column containing stock tickers
    
    Returns:
    pandas.DataFrame: DataFrame with ticker, converted ticker, and annual revenue growth
    """
    results = []
    
    for _, row in df.iterrows():
        original_ticker = row['Ticker']
        converted_ticker = convert_ticker_format(original_ticker)
        
        try:
            # Fetch the stock data
            stock = yf.Ticker(converted_ticker)
            
            # Get the annual financial data
            financials = stock.financials
            
            if financials.empty or 'Total Revenue' not in financials.index:
                results.append((original_ticker, None))
                continue
            
            # Extract revenue data for the last two years
            revenue = financials.loc['Total Revenue']
            
            if len(revenue) < 2:
                results.append((original_ticker, None))
                continue
            
            # Calculate revenue growth
            last_year_revenue = revenue.iloc[0]
            previous_year_revenue = revenue.iloc[1]
            
            if previous_year_revenue != 0:
                growth = (last_year_revenue - previous_year_revenue) / previous_year_revenue * 100
            else:
                growth = float('inf')  # Handle division by zero
            
            results.append((original_ticker, growth))
        except Exception as e:
            results.append((original_ticker, converted_ticker, None))
            print(f"Error processing {original_ticker} ({converted_ticker}): {str(e)}")
    
    return pd.DataFrame(results, columns=['Original Ticker','Annual Revenue Growth (%)'])

def process_csv_tickers(input_file, output_file):
    """
    Read tickers from a CSV file, analyze their growth, and write results to a new CSV file.
    
    Args:
    input_file (str): Path to the input CSV file containing tickers
    output_file (str): Path to the output CSV file for results
    """
    try:
        # Read the input CSV file
        df = pd.read_csv(input_file)
        
        # Ensure the CSV has a 'Ticker' column
        if 'Ticker' not in df.columns:
            raise ValueError("Input CSV must have a 'Ticker' column")
        
        # Analyze stock growth
        result_df = analyze_stock_growth(df)
        
        # Write results to the output CSV file
        result_df.to_csv(output_file, index=False)
        
        print(f"Analysis complete. Results written to {output_file}")
    
    except Exception as e:
        print(f"An error occurred: {str(e)}")


In [95]:
# Example usage
input_csv = "input_tickers.csv"
output_csv = "output_growth_results.csv"

process_csv_tickers(input_csv, output_csv)

Analysis complete. Results written to output_growth_results.csv
